In [1]:
import duckdb
con = duckdb.connect("../olist.duckdb", read_only=True)

In [2]:
con.sql("DESCRIBE orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


In [3]:
con.sql("""
SELECT order_status, COUNT(*) AS pedidos,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM orders GROUP BY 1 ORDER BY 2 DESC
""").df()

,order_status,pedidos,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [4]:
con.sql("""
SELECT COUNT(*) AS total,
  COUNT(*) - COUNT(order_approved_at)             AS sem_aprovacao,
  COUNT(*) - COUNT(order_delivered_carrier_date)  AS sem_data_transportadora,
  COUNT(*) - COUNT(order_delivered_customer_date) AS sem_data_entrega,
  COUNT(*) - COUNT(order_estimated_delivery_date) AS sem_data_prometida
FROM orders
""").df()

,total,sem_aprovacao,sem_data_transportadora,sem_data_entrega,sem_data_prometida
0,99441,160,1783,2965,0


In [5]:
con.sql("""
SELECT order_status, COUNT(*) AS pedidos
FROM orders WHERE order_delivered_customer_date IS NULL
GROUP BY 1 ORDER BY 2 DESC
""").df()

,order_status,pedidos
0,shipped,1107
1,canceled,619
2,unavailable,609
3,invoiced,314
4,processing,301
5,delivered,8
6,created,5
7,approved,2


In [6]:
con.sql("""
SELECT
  (SELECT COUNT(*) FROM orders)                              AS pedidos,
  (SELECT COUNT(DISTINCT customer_id) FROM customers)        AS customer_id_unicos,
  (SELECT COUNT(DISTINCT customer_unique_id) FROM customers) AS clientes_reais,
  (SELECT COUNT(*) FROM order_items)                         AS itens,
  (SELECT COUNT(DISTINCT order_id) FROM order_items)         AS pedidos_com_itens
""").df()

,pedidos,customer_id_unicos,clientes_reais,itens,pedidos_com_itens
0,99441,99441,96096,112650,98666


In [7]:
con.sql("""
SELECT
  SUM(CASE WHEN order_delivered_customer_date < order_purchase_timestamp     THEN 1 ELSE 0 END) AS entrega_antes_da_compra,
  SUM(CASE WHEN order_delivered_customer_date < order_delivered_carrier_date THEN 1 ELSE 0 END) AS entrega_antes_da_transportadora,
  SUM(CASE WHEN order_estimated_delivery_date < order_purchase_timestamp     THEN 1 ELSE 0 END) AS prazo_antes_da_compra
FROM orders
""").df()

,entrega_antes_da_compra,entrega_antes_da_transportadora,prazo_antes_da_compra
0,0.0,23.0,0.0
